[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DSamuelHodge/paper2code/blob/main/papers/weight-compiled-circuits/2611.26450-cpu-inside-an-llm/notebook.ipynb)

# Paper2Code: *A CPU Inside an LLM: Zero-Gradient Compilation of Exact Circuits into Pretrained Weights*

**Paper:** Tanmay Sah. *A CPU Inside an LLM: Zero-Gradient Compilation of Exact Circuits into Pretrained Weights.* arXiv:2611.26450v1 [cs.AI], 19 Nov 2026. [abs](https://arxiv.org/abs/2611.26450) · [pdf](https://arxiv.org/pdf/2611.26450) · [alphaXiv](https://www.alphaxiv.org/abs/2611.26450)

**Is there official code?** No public repository as of 23 Nov 2026. The paper describes a "release codebase" (`demo_minimal.py`, `patches/*.safetensors`, `src/circuits/arithmetic_auto/`, `src/sidecar/sparse_sidecar_runtime.py`), but it gives no URL. The arXiv listing has no code link, Hugging Face Papers lists no GitHub repo or project page, and the author's GitHub and Hugging Face accounts have no matching repository. I also found no unofficial reimplementation. This notebook is written from scratch from the PDF.

**What this notebook does.** It rebuilds the paper's analytical primitives in plain PyTorch, where you can test each one. It then **grafts a real, exact circuit (single-digit `a+b=`) into the weights' computation of a real Qwen3.5 model** with zero gradients and zero training examples, and measures how much the model's ordinary language behaviour changes. The default model is **`Qwen/Qwen3.5-9B` in 4-bit**. 📄 The paper's own Qwen model is **Qwen 3.5 4B**; set `MODEL_SIZE = "4B"` to use it. Set `MODEL_SIZE = "small"` for `Qwen/Qwen3.5-0.8B` on CPU.

**How to read the labels**
- 📄 **Paper:** stated in the PDF. Numbers are quoted from it, with the table, figure or theorem.
- 🧪 **Simplification:** a choice this notebook makes to stay small and readable. It is not the paper's setup.
- 💭 **Interpretation:** my reading where the PDF leaves details open.

> Scope. 📄 The paper compiles 4–8 digit signed {+, −, ×} circuits, Boolean and Majority/Parity circuits into Granite 4.2 3B and Qwen 3.5 4B, using up to K = 2,555 extra wires and a 5-stage compiler. 🧪 Here we build the same **kinds** of parts (Mellin-GLU steps, self-sink routing, memoryless shifts, width extension with norm compensation, logit emission). We assemble them into a **small** circuit: 1-digit addition with guards, K = 65 wires, about 1,700 neurons. Nothing here reproduces the paper's headline numbers; those are quoted in the appendix table.

## Contents
0. Setup and configuration
1. The idea in plain English
2. Primitives, each as an equation followed by code (pure PyTorch, no model needed)
   - 2.1 The problem: RMSNorm multiplies every token by an unknown λ
   - 2.2 Single-layer impossibility
   - 2.3 The Mellin-GLU identity and its discrete quadrature (Thm 3.2)
   - 2.4 The two-layer scale-invariant step
   - 2.5 Toy experiment A: a 6-digit ripple-carry adder that survives random λ in every layer
   - 2.6 Indefinite-RoPE self-sink attention (Thm 3.3.1)
   - 2.7 Memoryless GatedDeltaNet shifts (Thm 3.3.2)
3. Width extension on a real Qwen3.5
   - 3.1 Load the model; measure λ on real text
   - 3.2 Why not carve wires out of d? (Ky Fan bound, Thm 3.4.1)
   - 3.3 Width extension d → d + K with γ = √(d/(d+K)) (Thm 3.4.2)
4. Main method: compile and graft an exact `a+b=` circuit into Qwen3.5
   - 4.1 The circuit, layer by layer
   - 4.2 Calibration from norm medians (Stage 5)
   - 4.3 Toy experiment B: all 100 single-digit sums, guards, mid-sentence use, false triggers
   - 4.4 Natural-language retention (KL), with and without norm compensation
5. Limitations: how this toy differs from the paper
6. Exercises
7. Appendix: numbers from the paper

## 0. Setup

**Runtime:** use a GPU for the 9B model (Runtime → Change runtime type). In 4-bit it needs about 7 GB of GPU memory, so a T4 is enough. Parts 2 and 3.2–3.3 are cheap. Part 4 runs short prompts with the KV cache off (see 🧪 in 4.1). Without a GPU, the notebook switches to the small model on its own. Everything runs on CPU in a few minutes with the small model.

In [ ]:
# Install dependencies (skipped if SKIP_INSTALL=1, e.g. when validated offline)
import os, sys, subprocess
if os.environ.get("SKIP_INSTALL") != "1":
    pkgs = ["transformers>=5.24", "accelerate", "scipy", "matplotlib"]
    try:
        import torch
        if torch.cuda.is_available():
            pkgs.append("bitsandbytes>=0.45")
    except ImportError:
        pkgs.append("torch")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=True)
print("ok")

In [ ]:
# ====== CONFIGURATION (edit here) ======
import os, math, time, torch
import torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

MODEL_SIZE = os.environ.get("MODEL_SIZE", "9B")   # "9B"    -> Qwen/Qwen3.5-9B in 4-bit (GPU)
                                                   # "4B"    -> Qwen/Qwen3.5-4B, the paper's own Qwen model (GPU, bf16)
                                                   # "small" -> Qwen/Qwen3.5-0.8B (CPU / free tier)
MODEL_IDS = {"9B": "Qwen/Qwen3.5-9B", "4B": "Qwen/Qwen3.5-4B", "small": "Qwen/Qwen3.5-0.8B"}
HAS_GPU = torch.cuda.is_available()
if MODEL_SIZE != "small" and not HAS_GPU:
    print("No GPU found -> switching to the small model.")
    MODEL_SIZE = "small"
MODEL_ID = MODEL_IDS[MODEL_SIZE]
LOAD_4BIT = (MODEL_SIZE == "9B") and HAS_GPU
if HAS_GPU:
    DEVICE = "cuda"; DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    DEVICE = "cpu"; DTYPE = torch.float32   # 🧪 fp32 on CPU: exact comparisons, and bf16 on CPU is slow
if os.environ.get("FORCE_DTYPE"): DTYPE = getattr(torch, os.environ["FORCE_DTYPE"])   # e.g. test the bf16 path on CPU
torch.set_grad_enabled(False)   # 📄 zero gradients: every weight below is written in closed form
torch.manual_seed(0); np.random.seed(0)
print(dict(MODEL_ID=MODEL_ID, LOAD_4BIT=LOAD_4BIT, DEVICE=DEVICE, DTYPE=str(DTYPE)))

## 1. The idea in plain English

Language models are unreliable at exact multi-digit arithmetic. The usual fixes are **fine-tuning** (gradients and data) or **tools** (pause decoding, call a calculator, paste the result back). 📄 The paper takes a third route: **write an exact circuit directly into the pretrained weights**, in closed form, with **zero gradient steps, zero training examples and zero runtime hooks**. The model then computes `123456+654321=` with its own layers and keeps writing normal text afterwards.

Why is that hard? A pretrained Transformer is not a clean computer:
1. **RMSNorm rescales every token by an unknown factor λ** before every sublayer. 📄 The paper puts the swing at more than 250× across tokens and layers. A circuit that reads "the value on wire 7" sees λ·value with a λ it does not know.
2. **Modern MLPs are bias-free GLUs (SwiGLU).** With no bias you cannot write a threshold like "is x ≥ 9.5?" in one layer (Section 2.2).
3. **Attention and linear attention mix tokens with learned positional tricks** (RoPE, GatedDeltaNet decay), so copying a digit from 3 tokens back is not free.
4. **Every extra wire changes the norm** seen by the model's own features. Carving wires out of the existing d dimensions damages the model (a Ky Fan bound).

📄 The paper's answers, which this notebook rebuilds one by one:
- **Mellin-GLU quadrature.** A sum of SiLU-gated units on a log-spaced grid computes the **ratio** ṽ/r̃ = (λx)/(λr₀) = x/r₀. The unknown λ cancels, because it only *shifts* the grid. Two such layers make an exact, scale-invariant step function.
- **Indefinite-RoPE self-sink attention** with signed weights, and **memoryless GatedDeltaNet shifts**, move values between tokens while cancelling each token's λ.
- **Width extension.** Add K new wires (d → d + K) with zero coupling to the model's own weights, and rescale the old norm weights by γ = √(d/(d+K)) so the model's features see the same scale as before.
- **Emission.** Huge, exact logit bumps on the right digit tokens make the model *say* the answer.

📄 Results: on 6-digit arithmetic (N = 1,200, BF16), Granite 4.2 3B reaches 99.17% and Qwen 3.5 4B 100.00% ALU-prefix accuracy (100% with an FP32 sidecar), with 98%–100% mid-sentence resumption. A 12.8K-example LoRA reaches 42.2% / 61.7% and 0% inline continuation (Abstract, Tab. 4).

## 2. Primitives

### 2.1 The problem: RMSNorm multiplies each token by an unknown λ

Qwen3.5's norm (📄 Qwen uses a $(1+w)$ parametrisation) is

$$\mathrm{RMSNorm}(h)_i = \frac{h_i}{\sqrt{\tfrac{1}{D}\sum_j h_j^2+\epsilon}}\,(1+w_i) \;=\; \lambda\,h_i\,(1+w_i), \qquad \lambda = \Big(\tfrac{1}{D}\|h\|^2+\epsilon\Big)^{-1/2}.$$

A circuit wire carrying a value $x$ (with $1+w_i = 1$) is seen by the next sublayer as $\tilde x = \lambda x$. 📄 The paper therefore always carries a **reference wire** $r_0$ next to each payload, so that the sublayer sees the pair $(\tilde r, \tilde x) = (\lambda r_0, \lambda x)$, and λ is shared by both.

In [ ]:
def rmsnorm_lambda(h, eps=1e-6):
    return torch.rsqrt(h.pow(2).mean(-1, keepdim=True) + eps)

# Two tokens whose "feature" parts differ in size: the same circuit pair (r0, x) is seen at very different scales.
d, r0, x = 64, 1.0, 7.0
for feat_scale in [0.05, 1.0, 12.5]:
    h = torch.cat([feat_scale * torch.randn(d), torch.tensor([r0, x])])
    lam = rmsnorm_lambda(h).item()
    print(f"feature rms ≈ {feat_scale:5.2f}  ->  λ = {lam:8.3f}   (r̃, x̃) = ({lam*r0:7.3f}, {lam*x:7.3f})   ratio x̃/r̃ = {x:.1f}")

### 2.2 Single-layer impossibility (📄 Thm 3.2.1)

A bias-free GLU layer is $F(u) = W_d\,\big(\mathrm{SiLU}(W_g u)\odot W_u u\big)$. Because there is no bias, scaling the input by λ gives

$$\|F(\lambda u)\| \le C\,\lambda^2\|u\|^2 \to 0 \quad (\lambda \to 0),$$

so $F$ cannot output "1" for every scaled copy $\lambda u$ of an input that is above a threshold. 📄 "For any bias-free GLU ... $\|F_1(\lambda u)\| \to 0$": **one layer cannot build a scale-invariant step.** The code builds a random bias-free SwiGLU and watches its output vanish as λ shrinks.

In [ ]:
class BiasFreeGLU(nn.Module):
    def __init__(self, din, dh, dout):
        super().__init__()
        self.g = nn.Linear(din, dh, bias=False); self.u = nn.Linear(din, dh, bias=False); self.d = nn.Linear(dh, dout, bias=False)
    def forward(self, x): return self.d(F.silu(self.g(x)) * self.u(x))

glu = BiasFreeGLU(2, 256, 1)
u = torch.tensor([1.0, 7.0])           # (r0, x): "x = 7 is above a threshold"
lams = torch.logspace(-3, 1, 9)
for lam in lams:
    print(f"λ = {lam.item():8.4f}   |F(λu)| = {glu(lam * u).abs().item():.3e}")
print("-> quadratic decay to 0 as λ -> 0: no single bias-free GLU can be a step function at every scale.")

### 2.3 The Mellin-GLU identity and its discrete quadrature (📄 Thm 3.2)

The key identity uses $\int_0^\infty \frac{t}{e^t+1}\,dt = \frac{\pi^2}{12}$. Note that $\mathrm{SiLU}(-t)\cdot(-t) = \frac{t^2}{1+e^{t}}$ for $t>0$. Substitute $t = e^{u}\tilde r$ (so $dt = t\,du$), and multiply the second factor by $\tilde v/\tilde r$:

$$\frac{12}{\pi^2}\int_{-\infty}^{\infty} \mathrm{SiLU}\!\big(-e^{u}\tilde r\big)\,\big(-e^{u}\tilde v\big)\,du \;=\; \frac{\tilde v}{\tilde r}\qquad(\tilde r>0).$$

Replacing $\tilde r \to \lambda \tilde r$ only **shifts** $u$ by $\ln\lambda$, and the integral over all $u$ does not change. That is why λ cancels. 📄 The discrete version is one SwiGLU layer: gate row $-e^{k\Delta}$ on the reference wire, up row $-e^{k\Delta}$ on the payload, down weight $12\Delta/\pi^2$:

$$Q_{\Delta,K_-,K_+}(\tilde r,\tilde v) = \frac{12\Delta}{\pi^2}\sum_{k=K_-}^{K_+}\mathrm{SiLU}\!\big(-e^{k\Delta}\tilde r\big)\big(-e^{k\Delta}\tilde v\big).$$

📄 The paper uses $\Delta = 0.55$ and bounds the discretisation error by $\varepsilon_{\rm disc}(0.55) < 6.26\times10^{-6}$. There is an extra tail error when the grid $[K_-, K_+]$ does not cover the λ range.

In [ ]:
PI2 = math.pi ** 2
DELTA = 0.55   # 📄 grid spacing used in the paper

# (a) the constant: ∫ t/(e^t+1) dt = π²/12   (📄 "Iϕ for SiLU (0.8225)")
t = torch.linspace(1e-8, 60, 2_000_001, dtype=torch.float64)
print("∫ t/(e^t+1) dt ≈", torch.trapezoid(t / (torch.exp(t) + 1), t).item(), "  π²/12 =", PI2 / 12)

def mellin_Q(r, v, delta=DELTA, kmin=-40, kmax=40, dtype=torch.float64):
    k = torch.arange(kmin, kmax + 1, dtype=dtype)
    a = torch.exp(k * delta)                                   # gate grid α_k = e^{kΔ}
    r, v = torch.as_tensor(r, dtype=dtype)[..., None], torch.as_tensor(v, dtype=dtype)[..., None]
    return (12 * delta / PI2) * (F.silu(-a * r) * (-a * v)).sum(-1)

# (b) discretisation error at Δ=0.55: sweep the grid phase (λ shifts the grid continuously)
lam = torch.logspace(-3, 3, 4001, dtype=torch.float64)
q = mellin_Q(lam * 1.0, lam * 1.0)              # payload/reference = 1
print(f"max |Q - 1| over λ ∈ [1e-3, 1e3] = {(q - 1).abs().max().item():.2e}   (📄 bound ε_disc(0.55) < 6.26e-6)")
for dlt in [0.3, 0.55, 0.8, 1.2]:
    print(f"   Δ = {dlt:4.2f}: max |Q-1| = {(mellin_Q(lam, lam, delta=dlt, kmin=int(-25/dlt), kmax=int(25/dlt)) - 1).abs().max().item():.2e}")

Now the comparison in 📄 Fig. 1(a): a **single-scale** gate tuned so that it reads $x/r_0$ correctly at λ = 1 collapses or saturates as λ moves, while the Mellin grid stays flat.

In [ ]:
x_true = 7.0
lam = torch.logspace(math.log10(1/16), math.log10(16), 200, dtype=torch.float64)   # 256x swing (📄 "> 250x")
q_mellin = mellin_Q(lam * 1.0, lam * x_true, kmin=-20, kmax=20)
# Single-scale gate: one unit SiLU(-a r̃)(-a ṽ), normalised so it is exact at λ = 1
a0 = 1.0
single = F.silu(-a0 * lam) * (-a0 * lam * x_true) / (F.silu(torch.tensor(-a0, dtype=torch.float64)) * (-a0))
plt.figure(figsize=(6, 3.2))
plt.semilogx(lam, q_mellin, label="Mellin grid Q (41 units, Δ=0.55)")
plt.semilogx(lam, single, "--", label="single-scale gate (exact at λ=1)")
plt.axhline(x_true, color="k", lw=0.5); plt.ylim(-1, 20); plt.xlabel("unknown norm factor λ"); plt.ylabel("readout of x = 7")
plt.legend(); plt.title("Scale invariance (cf. 📄 Fig. 1a)"); plt.show()
print("Mellin max error over 256x swing:", (q_mellin - x_true).abs().max().item())
print("single-scale readout at λ=1/16, 1, 16:", single[0].item(), single[len(lam)//2].item(), single[-1].item())

### 2.4 The two-layer scale-invariant step (📄 Thm 3.2, Stages 1–2)

To decide "is $x \ge \tau$?" without biases, the paper composes two layers:

**Stage 1 (layer ℓ):** $\;p = \gamma\,\mathrm{SiLU}\big(\beta(\tilde x-\tau\tilde r)\big)\,\big(\beta(\tilde x-\tau\tilde r)\big)$

**Stage 2 (layer ℓ+1):** $\;H \approx Q_{\Delta,K_-,K_+}(\tilde p,\tilde p)$

With $z := \beta\lambda_{\min}\delta_{\rm sep} r_0$: if $x \ge (\tau+\delta_{\rm sep})r_0$ then $p>0$ is "large", and $Q(\tilde p,\tilde p) = \tilde p/\tilde p = 1$ up to $\varepsilon_{\rm disc}+\varepsilon_{\rm tail}$. If $x \le (\tau-\delta_{\rm sep})r_0$, then $p \approx \gamma z^2 e^{-z}$ is tiny, below the Mellin grid's window, so $Q \approx 0$. 📄 A gate offset that is ≤ −z when inactive gives **bit-exact −0.0** (for z ≥ 89 with direct IEEE kernels, z ≥ 104 in FP32 range-split kernels; the compiled S = 256 gives z ≈ 307).

In [ ]:
def window(lo, hi, delta=DELTA, t_lo=1e-3, t_hi=40.0):
    # grid [K-, K+] so that e^{kΔ} r̃ spans [t_lo, t_hi] for every r̃ in [lo, hi]  (💭 our tail choice)
    return math.floor(math.log(t_lo / hi) / delta), math.ceil(math.log(t_hi / lo) / delta)

def two_layer_step(x, tau, lam1, lam2, lam_min=1/16, lam_max=16, r0=1.0, zmin=60.0, xmax=20.0):
    # 🧪 float32, like the real circuit; λ1, λ2 = the unknown norm factors seen by stage 1 and stage 2
    x, lam1, lam2 = [torch.as_tensor(v, dtype=torch.float32) for v in (x, lam1, lam2)]
    beta = 2 * zmin / (lam_min * r0)                      # z = β λmin (δsep = 0.5) r0 = zmin
    umax = beta * lam_max * (xmax + abs(tau)) * r0
    gamma = 1.0 / umax ** 2                               # keeps p ≤ 1 so the wire barely moves the norm
    s = beta * (lam1 * x - tau * lam1 * r0)
    p = gamma * F.silu(s) * s                             # Stage 1
    kmin, kmax = window(gamma * zmin**2 * lam_min, lam_max)
    pt = lam2 * p
    k = torch.arange(kmin, kmax + 1, dtype=torch.float32); a = torch.exp(k * DELTA)
    return (12 * DELTA / PI2) * (F.silu(-a * pt[..., None]) * (-a * pt[..., None])).sum(-1)   # Stage 2: Q(p̃, p̃)

xs = torch.arange(0, 20, dtype=torch.float32)
g = torch.Generator().manual_seed(0)
worst_on, worst_off = 0.0, 0.0
for trial in range(2000):
    l1, l2 = torch.exp(torch.empty(2).uniform_(math.log(1/16), math.log(16), generator=g))
    H = two_layer_step(xs, 9.5, l1, l2)
    worst_on = max(worst_on, (H[xs >= 10] - 1).abs().max().item()); worst_off = max(worst_off, H[xs <= 9].abs().max().item())
print(f"H(x - 9.5) over 2000 random (λ1, λ2) ∈ [1/16, 16]²:  max |H-1| above = {worst_on:.2e},  max |H| below = {worst_off:.2e}")

### 2.5 Toy experiment A: a 6-digit adder that survives a random λ in every layer

🧪 A "residual machine": a residual vector of wires; every layer sees $\lambda_\ell \cdot h$ with a **fresh random** $\lambda_\ell \in [1/16, 16]$ (a 256× swing) and adds a bias-free SwiGLU output back. Wires: reference $r$, digits $a_i, b_i$, carries $c_i$ and outputs $o_i$. 📄 Exact carries come from the step at $\tau = 10m - 0.5$ (Thm 3.2.3). Here $m = 1$: $c_{i+1} = H(a_i+b_i+c_i - 9.5)$. The output digit $o_i = Q(\tilde r, \tilde a_i+\tilde b_i+\tilde c_i-10\tilde c_{i+1})$ is a Mellin ratio readout. The carry ripples, so the depth is 2 layers per digit plus 1 (📄 the paper's compiler builds TC0, constant-depth circuits instead of ripple chains).

The ablation replaces every Mellin grid with a **single-scale** unit that is exact only at λ = 1 (📄 the paper's "single-scale SwiGLU" ablation gives Arith 8.3% / 20.0%, Tab. 5).

In [ ]:
class ResidualMachine:
    # Wires: 0 = r; then a_0..a_{n-1}, b_*, c_0..c_n, p_* (stage-1 scratch), o_*
    def __init__(self, n=6, lam_min=1/16, lam_max=16, single_scale=False):
        self.n, self.lo, self.hi, self.single = n, lam_min, lam_max, single_scale
        self.idx = {"r": 0}; c = 1
        for nm in ["a", "b", "c", "p", "o"]:
            for i in range(n + 1 if nm == "c" else n): self.idx[f"{nm}{i}"] = c; c += 1
        self.D = c
    def lam(self, g): return math.exp(torch.empty(1).uniform_(math.log(self.lo), math.log(self.hi), generator=g).item())
    def Q(self, rt, vt, lo, hi):
        if self.single:   # one unit, exact at λ = 1 only
            return F.silu(-rt) * (-vt) / (F.silu(torch.tensor(-1.0)) * -1.0)
        kmin, kmax = window(lo, hi); a = torch.exp(torch.arange(kmin, kmax + 1, dtype=torch.float32) * DELTA)
        return (12 * DELTA / PI2) * (F.silu(-a * rt[..., None]) * (-a * vt[..., None])).sum(-1)
    def run(self, A, B, g):
        n, I = self.n, self.idx; N = A.shape[0]
        h = torch.zeros(N, self.D); h[:, I["r"]] = 1.0
        for i in range(n): h[:, I[f"a{i}"]] = A[:, i].float(); h[:, I[f"b{i}"]] = B[:, i].float()
        zmin, xmax = 60.0, 19.0
        beta = 2 * zmin / self.lo; umax = beta * self.hi * (xmax + 9.5); gamma = 1.0 / umax ** 2
        for i in range(n):
            # layer: stage 1 for carry c_{i+1}
            lam = self.lam(g); ht = lam * h
            s = beta * (ht[:, I[f"a{i}"]] + ht[:, I[f"b{i}"]] + ht[:, I[f"c{i}"]] - 9.5 * ht[:, I["r"]])
            h[:, I[f"p{i}"]] += gamma * F.silu(s) * s
            # layer: stage 2 -> carry; (single-scale ablation: one unit normalised at a typical p)
            lam = self.lam(g); ht = lam * h; pt = ht[:, I[f"p{i}"]]
            if self.single:
                p_ref = gamma * (beta * 0.5) ** 2   # p at λ = 1 for the smallest active margin
                h[:, I[f"c{i+1}"]] += self.Q(pt / p_ref, pt / p_ref, 0, 0)
            else:
                h[:, I[f"c{i+1}"]] += self.Q(pt, pt, gamma * zmin ** 2 * self.lo, self.hi)
        lam = self.lam(g); ht = lam * h     # final layer: all digit readouts in parallel
        for i in range(n):
            v = ht[:, I[f"a{i}"]] + ht[:, I[f"b{i}"]] + ht[:, I[f"c{i}"]] - 10 * ht[:, I[f"c{i+1}"]]
            h[:, I[f"o{i}"]] += self.Q(ht[:, I["r"]], v, self.lo, self.hi)
        return h[:, [I[f"o{i}"] for i in range(n)] + [I[f"c{n}"]]]

def digits(x, n): return torch.stack([(x // 10**i) % 10 for i in range(n)], 1)
g = torch.Generator().manual_seed(1); N = 1200   # 📄 the paper's 6-digit evaluation also uses N = 1,200
A = torch.randint(0, 10**6, (N,), generator=g); B = torch.randint(0, 10**6, (N,), generator=g)
for single in [False, True]:
    rm = ResidualMachine(6, single_scale=single)
    accs, errs = [], []
    for trial in range(5):   # 5 independent draws of the per-layer λ's (same λ for the whole batch within a draw)
        out = rm.run(digits(A, 6), digits(B, 6), torch.Generator().manual_seed(100 + trial))
        pred = (out.round().clamp(0, 9).long() * torch.tensor([10**i for i in range(7)])).sum(1)
        accs.append((pred == A + B).float().mean().item()); errs.append((out - out.round()).abs().max().item())
    print(f"{'single-scale (ablation)' if single else 'Mellin two-layer steps   '}: exact-sum accuracy per λ-draw = "
          f"{[f'{a:.1%}' for a in accs]}, max |wire - integer| = {max(errs):.2e}")

### 2.6 Indefinite-RoPE self-sink attention (📄 Thm 3.3.1, Fig. 1b)

To copy a value from the token $d_0$ positions back, a query must put all of its attention on that key. With RoPE, the query–key score between positions $t$ and $u$ is a **cosine kernel** of the offset $\delta$:

$$S(\delta) = \sum_m w_m \cos(\delta\,\omega_m), \qquad \omega_m = \theta^{-2m/d_h}.$$

📄 With $\theta_{\rm RoPE} = 10^7$ and 64 frequencies, **positive weights** leave $S(\delta) \ge +0.61$: far tokens still get large positive scores and leak attention. **Signed** weights, applied as $k_m = \mathrm{sgn}(w_m)\sqrt{|w_m|}$ on keys and $\sqrt{|w_m|}$ on queries, achieve $S(0) = +1$ and $S(\delta) \le -0.355$ for $1 \le |\delta| \le 1024$. Inactive queries then "park" on zero-payload self-sinks. The value carries $V_s = c_s[\lambda_s, \lambda_s x_s]$, and the ratio $\hat v/\hat r$ cancels the source token's λ.

💭 The paper says "signed Chebyshev weights" but does not print them. 🧪 We find weights with a linear program instead: maximise the margin μ subject to $\sum w = 1$, $S(\delta) \le -\mu$, and an ℓ1 budget $\sum|w| \le L$. We then report the smallest budget L that reaches the paper's −0.355.

In [ ]:
from scipy.optimize import linprog
D_HEAD, THETA, DMAX = 128, 1e7, 1024
om = THETA ** (-2 * np.arange(D_HEAD // 2) / D_HEAD)      # 64 RoPE frequencies
deltas = np.arange(1, DMAX + 1); Cmat = np.cos(np.outer(deltas, om))
def best_kernel(L=None, positive=False):
    n = len(om)
    if positive:
        res = linprog(np.r_[np.zeros(n), -1], A_ub=np.hstack([Cmat, np.ones((DMAX, 1))]), b_ub=np.zeros(DMAX),
                      A_eq=np.r_[np.ones(n), 0][None], b_eq=[1], bounds=[(0, None)] * n + [(None, None)], method="highs")
        return res.x[:n]
    A_ub = np.vstack([np.hstack([Cmat, -Cmat, np.ones((DMAX, 1))]), np.r_[np.ones(2 * n), 0][None]])
    res = linprog(np.r_[np.zeros(2 * n), -1], A_ub=A_ub, b_ub=np.r_[np.zeros(DMAX), L],
                  A_eq=np.r_[np.ones(n), -np.ones(n), 0][None], b_eq=[1], bounds=[(0, None)] * (2 * n) + [(None, None)], method="highs")
    return res.x[:n] - res.x[n:2 * n]
w_pos = best_kernel(positive=True)
print(f"best POSITIVE weights: S(0) = {w_pos.sum():.3f}, max_δ S(δ) = {(Cmat @ w_pos).max():+.3f}   (📄 positive weights leave S ≥ +0.61)")
lo_L, hi_L = 1.0, 20.0
for _ in range(30):   # bisection on the ℓ1 budget
    mid = (lo_L + hi_L) / 2
    if (Cmat @ best_kernel(mid)).max() <= -0.355: hi_L = mid
    else: lo_L = mid
w_sgn = best_kernel(hi_L)
print(f"SIGNED weights (ℓ1 budget {hi_L:.2f}): S(0) = {w_sgn.sum():.3f}, max_δ S(δ) = {(Cmat @ w_sgn).max():+.3f}   (📄 target ≤ -0.355)")
dd = np.arange(0, DMAX + 1)
plt.figure(figsize=(6, 3)); plt.plot(dd, np.cos(np.outer(dd, om)) @ w_pos, label="positive w"); plt.plot(dd, np.cos(np.outer(dd, om)) @ w_sgn, label="signed w")
plt.axhline(0, c="k", lw=.5); plt.xscale("symlog"); plt.xlabel("offset δ"); plt.ylabel("S(δ)"); plt.legend(); plt.title("RoPE kernel (cf. 📄 Fig. 1b)"); plt.show()

Now real attention with these weights. Each query at position $t$ is rotated to $t - d_0$, so its score with key $u$ is $\kappa\,c_u\,S(t-d_0-u)$. Here $c_u \in \{0,1\}$ marks "active" tokens (e.g. digits), inactive keys are zero vectors (score 0), and values are $V_u = c_u[\lambda_u, \lambda_u x_u]$. The readout is $\hat v/\hat r$.

In [ ]:
def rope_rotate(vec_pairs, pos):           # vec_pairs: [..., 64, 2]; rotate pair m by angle pos·ω_m
    ang = torch.as_tensor(pos, dtype=torch.float64)[..., None] * torch.as_tensor(om)
    c, s = torch.cos(ang), torch.sin(ang)
    x, y = vec_pairs[..., 0], vec_pairs[..., 1]
    return torch.stack([c * x - s * y, s * x + c * y], -1)

def self_sink_copy(xvals, active, d0, w, kappa=40.0, seed=0):
    T = len(xvals); g = torch.Generator().manual_seed(seed)
    lam = torch.exp(torch.empty(T, dtype=torch.float64).uniform_(math.log(1/16), math.log(16), generator=g))  # source λ_s
    wt = torch.as_tensor(w)
    base = torch.zeros(len(om), 2, dtype=torch.float64); base[:, 0] = 1.0
    qv = (wt.abs().sqrt()[:, None] * base)                 # query  √|w_m|
    kv = (torch.sign(wt) * wt.abs().sqrt())[:, None] * base   # key  sgn(w_m)√|w_m|
    K = torch.stack([active[u] * rope_rotate(kv, u) for u in range(T)]).reshape(T, -1)
    V = torch.stack([active * lam, active * lam * xvals], 1)
    out = []
    for t in range(T):
        q = rope_rotate(qv, t - d0).reshape(-1)
        att = torch.softmax(kappa * (K[: t + 1] @ q), 0)      # causal
        r_hat, v_hat = att @ V[: t + 1]
        out.append((v_hat / r_hat).item() if r_hat > 1e-30 else float("nan"))
    return out

T, d0 = 40, 3
xvals = torch.randint(0, 10, (T,)).double(); active = (torch.rand(T) < 0.6).double()
for name, w in [("signed", w_sgn), ("positive", w_pos)]:
    out = self_sink_copy(xvals, active, d0, w)
    ok = [abs(out[t] - xvals[t - d0].item()) < 1e-6 for t in range(d0, T) if active[t - d0] == 1]
    print(f"{name:8s} kernel: exact copies of x[t-{d0}] on active sources: {sum(ok)}/{len(ok)}")

### 2.7 Memoryless GatedDeltaNet shifts (📄 Thm 3.3.2)

Qwen3.5 interleaves full attention with **GatedDeltaNet** linear attention. Its state update is

$$S_t = e^{g_t}S_{t-1} + \hat k_t\big(v_t - e^{g_t}S_{t-1}^\top \hat k_t\big)^\top\beta_t,\qquad g_t = -e^{A_{\log}}\,\mathrm{softplus}(a_t + b_\Delta).$$

📄 Setting $W_a = 0$ ($a_t = 0$), $A_{\log} = 10$ and $b_\Delta = 10$ gives $g_t = -e^{10}\ln(1+e^{10}) < -2.20\times10^5$, so $e^{g_t}$ is **exactly 0.0 in FP32/BF16**: the head forgets everything at every step. A causal Conv1d of width 4 (taps $\delta \in \{1,2,3\}$) then delivers $x_{t-\delta}$. Since the conv output passes through SiLU, payloads ride in **antisymmetric pairs** $(\pm\tfrac12 u)$ and are read back with $\mathrm{SiLU}(u) - \mathrm{SiLU}(-u) \equiv u$.

In [ ]:
A_log, b_dt, a_t = torch.tensor(10.0), torch.tensor(10.0), torch.tensor(0.0)
g_t = -torch.exp(A_log) * F.softplus(a_t + b_dt)
print(f"g_t = {g_t.item():.4e}   (📄 < -2.20e5)")
for dt in [torch.float32, torch.bfloat16, torch.float16]:
    print(f"   exp(g_t) in {str(dt):15s} = {torch.exp(g_t.to(dt)).item()}")

# antisymmetric SiLU pair
u = torch.linspace(-50, 50, 10001)
print("max |SiLU(u) - SiLU(-u) - u| =", (F.silu(u) - F.silu(-u) - u).abs().max().item())

# A toy gated-delta recurrence with e^g = 0: the output at t depends only on token t (memoryless)
def gated_delta(q, k, v, beta, g):
    S = torch.zeros(k.shape[-1], v.shape[-1]); outs = []
    for t in range(q.shape[0]):
        kt = F.normalize(k[t], dim=0); qt = F.normalize(q[t], dim=0)
        S = torch.exp(g) * S + torch.outer(kt, (v[t] - torch.exp(g) * S.T @ kt) * beta)
        outs.append(S.T @ qt)
    return torch.stack(outs)
T, dk = 12, 8
q, k, v = torch.randn(T, dk), torch.randn(T, dk), torch.randn(T, dk)
o_full = gated_delta(q, k, v, 1.0, g_t)
o_alone = torch.stack([gated_delta(q[t:t+1], k[t:t+1], v[t:t+1], 1.0, g_t)[0] for t in range(T)])
print("memoryless: max |out(full sequence) - out(token alone)| =", (o_full - o_alone).abs().max().item())

# causal conv taps deliver x_{t-δ}: a width-4 depthwise conv whose kernel is a one-hot at tap δ
x = torch.randn(1, 1, 10)
for dl in [1, 2, 3]:
    w = torch.zeros(1, 1, 4); w[0, 0, 3 - dl] = 1.0
    y = F.conv1d(F.pad(x, (3, 0)), w)
    print(f"tap δ={dl}: y[t] == x[t-{dl}] ->", torch.allclose(y[0, 0, dl:], x[0, 0, :-dl]), "  zero for t<δ ->", bool((y[0, 0, :dl] == 0).all()))

## 3. Width extension on a real Qwen3.5

### 3.1 Load the model and measure λ on real text

🧪 We load the text-only model (`AutoModelForCausalLM`). The 9B model is loaded in 4-bit NF4. 📄 This mirrors the paper's "Hybrid PTQ": $W_{\rm NL}$ quantised to INT8/INT4 while the circuit runs in an FP32 sidecar (Tab. 3).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
kwargs = dict(dtype=DTYPE, device_map=DEVICE)
if LOAD_4BIT:
    from transformers import BitsAndBytesConfig
    kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs).eval()
cfg = model.config
print(f"loaded {MODEL_ID} in {time.time()-t0:.0f}s | d = {cfg.hidden_size}, {cfg.num_hidden_layers} layers, "
      f"layer types (first 8): {cfg.layer_types[:8]}")

@torch.no_grad()
def generate(prompt, n=4):
    ids = tok(prompt, return_tensors="pt").input_ids.to(model.device)
    out = model.generate(ids, max_new_tokens=n, do_sample=False, use_cache=False)  # 🧪 cache off, see 4.1
    return tok.decode(out[0, ids.shape[1]:])

NEUTRAL = [  # 📄 Stage 5 calibrates from norm medians on short neutral English sentences ("Digit-Free News")
    "The weather was pleasant this morning.", "She walked to the library after lunch.", "Music filled the quiet hall.",
    "They planted trees along the river bank.", "A small cat slept on the warm windowsill.", "He wrote a long letter to his friend.",
    "The train arrived later than expected.", "Fresh bread smells wonderful."]
EVAL_TEXT = ("Scientists have long studied how birds navigate during migration. Some species appear to sense the "
             "magnetic field of the Earth, while others rely on the position of the sun and stars. In the autumn, "
             "flocks gather near the coast before crossing the sea, and many return to the same nesting site each spring.")

In [ ]:
@torch.no_grad()
def norm_lambdas(texts):
    # λ = 1/rms of the residual stream entering each norm, per token
    lams, hooks = {}, []
    lm = model.model
    def mk(name):
        def hook(mod, inp): lams.setdefault(name, []).append(torch.rsqrt(inp[0].float().pow(2).mean(-1) + mod.eps).flatten().cpu())
        return hook
    for i, L in enumerate(lm.layers):
        hooks.append(L.input_layernorm.register_forward_pre_hook(mk(f"L{i}.in")))
        hooks.append(L.post_attention_layernorm.register_forward_pre_hook(mk(f"L{i}.post")))
    hooks.append(lm.norm.register_forward_pre_hook(mk("final")))
    try:
        for s in texts: model(**tok(s, return_tensors="pt").to(model.device), use_cache=False)
    finally:
        for h in hooks: h.remove()
    return {k: torch.cat(v) for k, v in lams.items()}

lams = norm_lambdas(NEUTRAL + [EVAL_TEXT, "Compute 123456+654321= and then 7+8=15."])
allv = torch.cat(list(lams.values()))
print(f"λ over all tokens and norms: min {allv.min():.3g}, median {allv.median():.3g}, max {allv.max():.3g}  "
      f"-> swing {allv.max()/allv.min():.0f}x   (📄 the paper cites > 250x)")
names = list(lams.keys()); med = [lams[n].median().item() for n in names]
plt.figure(figsize=(7, 2.8)); plt.semilogy(med, ".-"); plt.xlabel("norm index (input, post, ... final)"); plt.ylabel("median λ"); plt.title("Per-norm median λ on real text"); plt.show()

### 3.2 Why not carve the K wires out of the existing d dimensions? (📄 Thm 3.4.1)

You could reserve K directions inside $\mathbb{R}^d$ with a projection $P_\perp = I_d - UU^\top$. 📄 Ky Fan's theorem says this always costs something:

$$\mathbb{E}\,\|h - P_\perp h\|_2^2 = \mathrm{tr}(U^\top\Sigma U) \;\ge\; \sum_{i=d-K+1}^{d}\sigma_i > 0,$$

where $\sigma_i$ are the eigenvalues of $\Sigma = \mathbb{E}[hh^\top]$. The best you can do is to remove the K least-energetic directions. 📄 In Fig. 3(a), SVD projection costs 43×–3,800× more next-token KL than width extension. The code measures the bound for the layer-0 residual stream (the token embeddings) of our model. It then applies the best projection there and measures the next-token KL on held-out text.

In [ ]:
K_DEMO = 65   # the number of wires our circuit in Part 4 will use
# 💭 Σ for the layer-0 residual stream: the hidden state there is just the token embedding, so we take Σ = E[e eᵀ]
# over the vocabulary (uniform weights, our choice). Unlike a few sentences, this Σ has full rank, as the theorem assumes.
E = model.model.embed_tokens.weight
Sigma = torch.zeros(E.shape[1], E.shape[1], dtype=torch.float64)
for i in range(0, E.shape[0], 16384):
    blk = E[i:i + 16384].float()
    Sigma += (blk.T @ blk).double().cpu()
Sigma /= E.shape[0]
evals, evecs = torch.linalg.eigh(Sigma)            # ascending: σ_d ≤ ... ≤ σ_1
share = evals[:K_DEMO].sum().item() / evals.sum().item()
print(f"vocab-level Σ: d = {E.shape[1]}, smallest eigenvalue {evals[0].item():.3e} > 0")
print(f"Ky Fan lower bound: any projection vacating K={K_DEMO} wires removes ≥ {share:.2e} of the expected energy E||h||²")
print(f"(a random K-dim subspace removes ≈ K/d = {K_DEMO / E.shape[1]:.2e})")

@torch.no_grad()
def logprobs(text):
    ids = tok(text, return_tensors="pt").input_ids.to(model.device)
    return model(input_ids=ids, use_cache=False).logits.float().log_softmax(-1)[0].cpu()
def kl(p_log, q_log): return (p_log.exp() * (p_log - q_log)).sum(-1).mean().item()
base_lp = logprobs(EVAL_TEXT)

# Project out K directions of the embedding output (layer-0 residual) and measure next-token KL on held-out text.
for which in ["weakest", "random"]:
    U = evecs[:, :K_DEMO] if which == "weakest" else torch.linalg.qr(torch.randn(E.shape[1], K_DEMO, dtype=torch.float64))[0]
    P = (torch.eye(E.shape[1], dtype=torch.float64) - U @ U.T).float().to(model.device)
    hk = model.model.embed_tokens.register_forward_hook(lambda m, i, o: (o.float() @ P).to(o.dtype))
    try: lp = logprobs(EVAL_TEXT)
    finally: hk.remove()
    print(f"projection P⊥ removing K={K_DEMO} {which:7s} directions at layer 0: KL(base || projected) = {kl(base_lp, lp):.5f}")
print("💭 Ky Fan bounds lost *energy*; low-energy directions can still carry signal the model uses, so the KL of the"
      " energy-optimal projection need not be the smallest.")
print("Compare with the width-extension KL in 4.4. 🧪 We project at layer 0 only; the paper's SVD-projection control vacates wires for the whole circuit.")

### 3.3 Width extension d → d + K with norm compensation (📄 Thm 3.4.2)

Instead, append K new wires: $h_{\rm ext} = [h_{\rm NL}; h_{\rm circ}] \in \mathbb{R}^{d+K}$, with **zero cross-coupling** (the model's weights $W_{\rm NL}$ never read or write circuit wires, and vice versa). The norm now averages over $d+K$ entries, which would enlarge every NL feature by $\sqrt{(d+K)/d}$. 📄 The fix is to scale the NL norm weights by $\gamma_{\rm NL} = \sqrt{d/(d+K)}$. For Qwen's $(1+w)$ norm (💭 our reading), the new effective weight is $\gamma(1+w)$. The NL features then see

$$\frac{h_{\rm NL}\sqrt{d+K}}{\sqrt{\|h_{\rm NL}\|^2 + \|h_{\rm circ}\|^2}}\cdot\sqrt{\tfrac{d}{d+K}}\,(1+w) = \frac{h_{\rm NL}\sqrt d}{\sqrt{\|h_{\rm NL}\|^2(1+\rho^2)}}(1+w),\qquad \rho = \frac{\|h_{\rm circ}\|}{\|h_{\rm NL}\|},$$

which matches the original norm up to $(1+\rho^2)^{-1/2}$. That is why circuit wires are kept tiny: 📄 $M_0 = 2^{-9}$ for Qwen ("Layer-0 drift < 0.015%"). 📄 Without compensation, Qwen's NL KL rises from 0.0008 to 2.2725 and ARC-C falls from 92.7% to 38.5% (Tab. 5). Uncompensated d + K "inflates KL by 547×" (Fig. 3a).

🧪 We implement the extension as a **sidecar**, in the spirit of 📄 the paper's factorized `SparseSidecar` (Tab. 3, Sec. G.4). The model's own modules keep width d and run on $h[:d]$, while the K circuit wires and their small fp32 weight blocks live beside them. This is mathematically the same as a block-diagonal $(d+K)$-wide weight with zero off-diagonal blocks.

In [ ]:
class ExtRMSNorm(nn.Module):
    # RMSNorm over d + K (+ pad inert zero wires). NL weight becomes γ(1+w); circuit weight is 1 (i.e. w = 0).
    def __init__(self, orig, d, K, compensate=True, pad=0):
        super().__init__()
        self.d, self.D, self.eps = d, d + K + pad, orig.eps
        gamma = math.sqrt(d / (d + K + pad)) if compensate else 1.0
        self.register_buffer("scale_nl", gamma * (1.0 + orig.weight.detach().float()))
        self.lam = None
    def forward(self, x):
        x = x.float()
        inv = torch.rsqrt(x.pow(2).sum(-1, keepdim=True) / self.D + self.eps)   # mean over d + K + pad entries
        self.lam = inv.detach()
        y = x * inv
        return torch.cat([y[..., :self.d] * self.scale_nl, y[..., self.d:]], -1)

class SideMixer(nn.Module):
    # token mixer (GatedDeltaNet or attention) on h[:d], plus an optional fp32 circuit shift on h[d:]
    def __init__(self, orig, d, dtype, shift=None):
        super().__init__(); self.orig, self.d, self.dtype, self.shift = orig, d, dtype, shift
    def forward(self, hidden_states, **kw):
        x = hidden_states
        if kw.get("position_embeddings") is not None:
            kw["position_embeddings"] = tuple(t.to(self.dtype) for t in kw["position_embeddings"])
        am = kw.get("attention_mask")
        if torch.is_tensor(am) and am.is_floating_point(): kw["attention_mask"] = am.to(self.dtype)
        out = self.orig(hidden_states=x[..., :self.d].to(self.dtype).contiguous(), **kw)
        nl = out[0] if isinstance(out, tuple) else out
        c = self.shift(x[..., self.d:]) if self.shift is not None else torch.zeros_like(x[..., self.d:])
        y = torch.cat([nl.float(), c], -1)
        return (y,) + tuple(out[1:]) if isinstance(out, tuple) else y

class SideMLP(nn.Module):
    def __init__(self, orig, d, dtype, circ=None):
        super().__init__(); self.orig, self.d, self.dtype, self.circ = orig, d, dtype, circ
    def forward(self, x):
        nl = self.orig(x[..., :self.d].to(self.dtype)).float()
        c = self.circ(x[..., self.d:]) if self.circ is not None else torch.zeros_like(x[..., self.d:])
        return torch.cat([nl, c], -1)

class SideEmbed(nn.Module):
    def __init__(self, orig, table):
        super().__init__(); self.orig = orig; self.register_buffer("table", table)   # [vocab, K] fp32 circuit embedding
    def forward(self, ids):
        return torch.cat([self.orig(ids).float(), self.table[ids]], -1)

class SideHead(nn.Module):
    # logits = W_lm h[:d] + Δz,  Δz = W_z h[d:]  (only digit-token rows of W_z are non-zero)
    def __init__(self, orig, d, dtype, Wz):
        super().__init__(); self.orig, self.d, self.dtype = orig, d, dtype; self.register_buffer("Wz", Wz)
        self.last_dz = None
    def forward(self, x):
        z = self.orig(x[..., :self.d].to(self.dtype)).float()
        dz = x[..., self.d:].to(self.Wz.device) @ self.Wz.T
        self.last_dz = dz.detach()
        return z + dz.to(z.device)
print("ok")

## 4. Main method: compile and graft an exact `a+b=` circuit into Qwen3.5

### 4.1 The circuit, layer by layer

📄 The paper's compiler maps a declarative arithmetic spec to wires and layers in 5 stages and emits Base-1000 BPE chunks. 🧪 We hand-compile one tiny program. When the model reads `a+b=` (single digits, with the token before `a` not a digit), it must emit the digits of a+b and then continue normally. Qwen's tokenizer splits digits, `+` and `=` into single tokens.

| where | what | primitive |
|---|---|---|
| embedding | wires $r = M_0$ on every token; flags `eq`, `plus`, `isd` $= M_0$; digit value $M_0\cdot d$ | 📄 reference wire, $M_0 = 2^{-9}$ |
| L0 token mixer | copy $(r, \text{dig}, \text{isd})$ from $t-1, t-3$, $(r,\text{plus})$ from $t-2$, $(r, \text{isd})$ from $t-4$ | 📄 memoryless shift (🧪 fp32 sidecar shift instead of reprogrammed GDN heads) |
| L0 MLP | $S = M_0(a+b)$ and a guard count via Mellin ratio readouts $Q(\tilde r_{t-\delta}, \tilde x_{t-\delta})$ | 📄 Mellin-GLU |
| L1 MLP | Stage 1 for thresholds $s - 0.5$, $s = 0..18$, gated off by $-100\cdot(\text{guard} < 4)$ | 📄 Stage 1 with complement gate |
| L2 MLP | Stage 2: $h_s = Q(\tilde p_s, \tilde p_s) \in \{0, 1\}$ | 📄 Stage 2 |
| a later GDN layer | shift the one-hot $I_{10+j} = h_{10+j} - h_{11+j}$ to $t+1$ | 📄 memoryless shift (🧪 sidecar) |
| LM head | at `=`: $+G$ on token `str(s)` if $s<10$, or on `1` if $s \ge 10$; at $t+1$: $+G'$ on the units digit | 📄 logit emission (🧪 one-sided bumps, not the paper's differential +254/−256 tied readout) |

🧪 **KV cache off.** The circuit shifts read earlier tokens' circuit wires directly, and we did not extend the KV/conv caches. So generation recomputes the whole prefix each step (`use_cache=False`).

In [ ]:
M0 = 2.0 ** -9    # 📄 circuit scale for Qwen ("M0 ∈ {2^-4, 2^-9}"; 2^-9 for Qwen)

class CircuitMLP:
    # bias-free SwiGLU over the K circuit wires, out = W_d (SiLU(W_g x) ⊙ W_u x), kept in fp32
    def __init__(self, K): self.K = K; self.g, self.u, self.d = [], [], []
    def neuron(self, gate_row, up_row, down_row):
        self.g.append(gate_row); self.u.append(up_row); self.d.append(down_row)
    def mellin(self, ref, pay, out, scale, kmin, kmax, delta=DELTA):
        # writes scale · Q(r̃_ref, Σ c ṽ) ≈ scale · (payload / reference) into wire `out`
        for k in range(kmin, kmax + 1):
            a = math.exp(k * delta)
            self.neuron({ref: -a}, {w: -a * c for w, c in pay.items()}, {out: scale * 12 * delta / PI2})
    def build(self, device):
        N, K = len(self.g), self.K
        Wg, Wu, Wd = torch.zeros(N, K, dtype=torch.float64), torch.zeros(N, K, dtype=torch.float64), torch.zeros(K, N, dtype=torch.float64)
        for n, (g, u, d) in enumerate(zip(self.g, self.u, self.d)):
            for w, c in g.items(): Wg[n, w] += c
            for w, c in u.items(): Wu[n, w] += c
            for w, c in d.items(): Wd[w, n] += c
        self.Wg, self.Wu, self.Wd = Wg.float().to(device), Wu.float().to(device), Wd.float().to(device)
        return self
    def nnz(self): return int((self.Wg != 0).sum() + (self.Wu != 0).sum() + (self.Wd != 0).sum())
    def __call__(self, x): return (F.silu(x @ self.Wg.T) * (x @ self.Wu.T)) @ self.Wd.T

class CircuitShift:
    # out[t, w] = Σ_i c_i x̃[t-δ, i]   (zero for t < δ): the effect of a memoryless GDN head + conv tap δ
    def __init__(self, K): self.K = K; self.taps = []
    def copy(self, delta, out, row): self.taps.append((delta, out, row))
    def __call__(self, x):
        out = torch.zeros_like(x)
        for delta, w, row in self.taps:
            v = sum(c * x[..., i] for i, c in row.items())
            out[..., delta:, w] += v[..., :-delta]
        return out

# ---- wire layout ----
WIRE = {}
for n in ["r", "eq", "plus", "isd", "dig", "s1_r", "s1_dig", "s1_isd", "s2_r", "s2_plus", "s3_r", "s3_dig", "s3_isd",
          "s4_r", "s4_isd", "S", "act"] + [f"p{s}" for s in range(19)] + [f"h{s}" for s in range(19)] + ["e_r"] + [f"e{j}" for j in range(9)]:
    WIRE[n] = len(WIRE)
K_WIRES, NS = len(WIRE), 19
print("K =", K_WIRES, "circuit wires")

In [ ]:
class Graft:
    def __init__(self, model, tok, compensate=True, pad=0, margin=30.0, emit_gain=1000.0):
        self.model, self.tok, self.margin, self.emit_gain = model, tok, margin, emit_gain
        lm = model.model; self.lm = lm; self.layers = lm.layers
        self.d = model.config.hidden_size; self.dtype = lm.embed_tokens.weight.dtype; self.dev = lm.embed_tokens.weight.device
        self.orig = dict(embed=lm.embed_tokens, norm=lm.norm, head=model.lm_head,
                         layers=[(L.input_layernorm, L.post_attention_layernorm, self._mixer(L), L.mlp) for L in self.layers])
        self.ids = {c: tok.convert_tokens_to_ids(c) for c in "0123456789+="}
        V = lm.embed_tokens.weight.shape[0]; K = K_WIRES
        table = torch.zeros(V, K); table[:, WIRE["r"]] = M0
        table[self.ids["="], WIRE["eq"]] = M0; table[self.ids["+"], WIRE["plus"]] = M0
        for c in "0123456789":
            table[self.ids[c], WIRE["isd"]] = M0; table[self.ids[c], WIRE["dig"]] = M0 * int(c)
        lm.embed_tokens = SideEmbed(self.orig["embed"], table.to(self.dev))
        for i, L in enumerate(self.layers):
            n_in, n_post, mixer, mlp = self.orig["layers"][i]
            L.input_layernorm = ExtRMSNorm(n_in, self.d, K, compensate, pad).to(n_in.weight.device)
            L.post_attention_layernorm = ExtRMSNorm(n_post, self.d, K, compensate, pad).to(n_post.weight.device)
            self._set_mixer(L, SideMixer(mixer, self.d, self.dtype))
            L.mlp = SideMLP(mlp, self.d, self.dtype)
        lm.norm = ExtRMSNorm(self.orig["norm"], self.d, K, compensate, pad).to(self.orig["norm"].weight.device)
        self.head_dev = self.orig["head"].weight.device
        self.head = SideHead(self.orig["head"], self.d, self.dtype, torch.zeros(V, K, device=self.head_dev)); model.lm_head = self.head
        self.shift_layer = next(i for i in range(4, len(self.layers)) if self.layers[i].block_type == "linear_attention")
    @staticmethod
    def _mixer(L): return L.linear_attn if L.block_type == "linear_attention" else L.self_attn
    @staticmethod
    def _set_mixer(L, m):
        if L.block_type == "linear_attention": L.linear_attn = m
        else: L.self_attn = m
    def remove(self):   # restore the untouched model
        self.lm.embed_tokens, self.lm.norm, self.model.lm_head = self.orig["embed"], self.orig["norm"], self.orig["head"]
        for L, (a, p, mix, mlp) in zip(self.layers, self.orig["layers"]):
            L.input_layernorm, L.post_attention_layernorm, L.mlp = a, p, mlp; self._set_mixer(L, mix)

    @torch.no_grad()
    def calibrate(self, sentences):
        # 📄 Stage 5: per-sublayer λ medians on short neutral sentences (no digits, no labels, no gradients)
        taps = {"a": self.layers[0].input_layernorm, "b": self.layers[0].post_attention_layernorm,
                "c": self.layers[1].post_attention_layernorm, "d": self.layers[2].post_attention_layernorm,
                "e": self.layers[self.shift_layer].input_layernorm, "f": self.lm.norm}
        vals = {k: [] for k in taps}
        for s in sentences:
            self.model(input_ids=self.tok(s, return_tensors="pt").input_ids.to(self.dev), use_cache=False)
            for k, m in taps.items(): vals[k].append(m.lam.flatten().cpu())
        self.lam_med = {k: torch.cat(v).median().item() for k, v in vals.items()}
        return self.lam_med

    def build(self):
        W, K, med, R = WIRE, K_WIRES, self.lam_med, self.margin
        lo = {k: v / R for k, v in med.items()}; hi = {k: v * R for k, v in med.items()}   # 🧪 ×/÷30 margin around medians
        # L0 mixer: memoryless shifts; scaled by 1/median(λ) so shifted copies stay ~M0 (they barely move the norm)
        ca = 1.0 / med["a"]; sh0 = CircuitShift(K)
        for dl, outs in [(1, [("s1_r", "r"), ("s1_dig", "dig"), ("s1_isd", "isd")]), (2, [("s2_r", "r"), ("s2_plus", "plus")]),
                         (3, [("s3_r", "r"), ("s3_dig", "dig"), ("s3_isd", "isd")]), (4, [("s4_r", "r"), ("s4_isd", "isd")])]:
            for o, i in outs: sh0.copy(dl, W[o], {W[i]: ca})
        self._mixer(self.layers[0]).shift = sh0
        # L0 MLP: Mellin readouts. S = a + b ; act = eq + isd(t-1) + plus(t-2) + isd(t-3) - isd(t-4)  (= 4 iff the pattern is valid)
        m0 = CircuitMLP(K); k0 = window(lo["b"] * min(1.0, ca * lo["a"]) * M0, hi["b"] * max(1.0, ca * hi["a"]) * M0)
        m0.mellin(W["s1_r"], {W["s1_dig"]: 1}, W["S"], M0, *k0); m0.mellin(W["s3_r"], {W["s3_dig"]: 1}, W["S"], M0, *k0)
        m0.mellin(W["r"], {W["eq"]: 1}, W["act"], M0, *k0)
        m0.mellin(W["s1_r"], {W["s1_isd"]: 1}, W["act"], M0, *k0); m0.mellin(W["s2_r"], {W["s2_plus"]: 1}, W["act"], M0, *k0)
        m0.mellin(W["s3_r"], {W["s3_isd"]: 1}, W["act"], M0, *k0); m0.mellin(W["s4_r"], {W["s4_isd"]: -1}, W["act"], M0, *k0)
        self.layers[0].mlp.circ = m0.build(self.dev)
        # L1 MLP: Stage 1, u_s = S - (s - 0.5) r + 100 (act - 4 r)   (complement gate: invalid -> u ≤ -81 λ M0 -> exact -0.0)
        m1 = CircuitMLP(K); zmin = 60.0; rc = hi["c"] / lo["c"]
        beta = 2 * zmin / (lo["c"] * M0); umax = 2 * zmin * 18.5 * rc; g1 = M0 / umax ** 2
        for s in range(NS):
            row = {W["S"]: beta, W["r"]: -beta * (s - 0.5) - 400 * beta, W["act"]: 100 * beta}
            m1.neuron(row, row, {W[f"p{s}"]: g1})
        self.layers[1].mlp.circ = m1.build(self.dev)
        # L2 MLP: Stage 2, h_s = Q(p̃_s, p̃_s)
        m2 = CircuitMLP(K); k2 = window(g1 * zmin ** 2 * lo["d"], M0 * hi["d"])
        for s in range(NS): m2.mellin(W[f"p{s}"], {W[f"p{s}"]: 1}, W[f"h{s}"], M0, *k2)
        self.layers[2].mlp.circ = m2.build(self.dev)
        # later GDN layer: shift the one-hot I_{10+j} forward by one token (units digit of sums ≥ 10)
        ce = 1.0 / med["e"]; sh = CircuitShift(K); sh.copy(1, W["e_r"], {W["r"]: ce})
        for j in range(9):
            row = {W[f"h{10+j}"]: ce}
            if 11 + j < NS: row[W[f"h{11+j}"]] = -ce
            sh.copy(1, W[f"e{j}"], row)
        self._mixer(self.layers[self.shift_layer]).shift = sh
        # emission: Δz on digit tokens; gains set so even the smallest plausible λ gives ≥ emit_gain nats
        V = self.head.Wz.shape[0]; Wz = torch.zeros(V, K)
        G1 = self.emit_gain / (lo["f"] * M0); G2 = self.emit_gain / (lo["f"] * ce * lo["e"] * M0)
        for s in range(NS):
            tid = self.ids[str(s)] if s < 10 else self.ids["1"]
            Wz[tid, W[f"h{s}"]] += G1
            if s + 1 < NS: Wz[tid, W[f"h{s+1}"]] -= G1
        for j in range(9): Wz[self.ids[str(j)], W[f"e{j}"]] += G2
        self.head.Wz = Wz.to(self.head_dev)
        nnz = m0.nnz() + m1.nnz() + m2.nnz() + int((Wz != 0).sum()) + len(sh0.taps) + len(sh.taps) + 5 + 12
        self.info = dict(K=K, d=self.d, gamma=round(math.sqrt(self.d / (self.d + K)), 5), mellin_window_L0=k0, mellin_window_L2=k2,
                         neurons=dict(L0=len(m0.g), L1=len(m1.g), L2=len(m2.g)), shift_layer=self.shift_layer,
                         beta=f"{beta:.3g}", gamma1=f"{g1:.3g}", nonzeros=nnz,
                         nonzero_share=("n/a (4-bit packed)" if getattr(self.model, "is_loaded_in_4bit", False)
                                        else f"{nnz / sum(p.numel() for p in self.model.parameters()):.2e}"))
        return self.info
print("ok")

### 4.2 Calibration from norm medians (📄 Stage 5)

📄 The compiler's last stage sets gains from norm medians on short neutral sentences: "C1: Digit-Free News (16 sents, 0 math)", or a 0-token weight-analytic estimate (C2). 🧪 We use 8 neutral sentences and a ×/÷30 safety margin around each median (💭 the margin is our choice). Before grafting, we record the base model's answers and log-probs so we can compare.

In [ ]:
PAIRS = [(a, b) for a in range(10) for b in range(10)]
t0 = time.time()
base_ans = {(a, b): generate(f"{a}+{b}=", 2 if a + b >= 10 else 1) for a, b in PAIRS}
base_acc = sum(base_ans[p] == str(sum(p)) for p in PAIRS)
print(f"BASE model: {base_acc}/100 exact on bare 'a+b=' prompts ({time.time()-t0:.0f}s). Misses:",
      [(p, base_ans[p]) for p in PAIRS if base_ans[p] != str(sum(p))][:10])
EXTRA = ["Bob had 7 apples and got 8 more, so 7+8=", "12+57=", "The code 4+4=8 is printed on the box, and 6+9="]
base_extra = {s: generate(s, 8) for s in EXTRA}
base_lp = logprobs(EVAL_TEXT)

graft = Graft(model, tok)
print("λ medians:", {k: round(v, 3) for k, v in graft.calibrate(NEUTRAL).items()})
print(graft.build())

### 4.3 Toy experiment B: all 100 single-digit sums, guards, mid-sentence use, false triggers

📄 Paper metrics: ALU-prefix accuracy, mid-sentence resumption, and **false triggers** on out-of-domain text (0/11,134 OOD triggers; "max |Δz_circ| = 0.0" on non-arithmetic text). We test the same three properties on our tiny circuit.

In [ ]:
t0 = time.time()
graft_ans = {(a, b): generate(f"{a}+{b}=", 2 if a + b >= 10 else 1) for a, b in PAIRS}
graft_acc = sum(graft_ans[p] == str(sum(p)) for p in PAIRS)
print(f"GRAFTED model: {graft_acc}/100 exact ({time.time()-t0:.0f}s).  Base was {base_acc}/100.")

# How decisive is the emission? logit margin of the correct first token at '='
ids = tok("8+7=", return_tensors="pt").input_ids.to(model.device)
with torch.no_grad(): z = model(input_ids=ids, use_cache=False).logits[0, -1].float()
top = z.topk(3)
print("8+7= top-3 next tokens:", [(tok.decode([i]), round(v, 1)) for v, i in zip(top.values.tolist(), top.indices.tolist())])

print("\nmid-sentence use and guards (graft vs base):")
for s in EXTRA:
    print(f"  {s!r}\n     graft -> {generate(s, 8)!r}\n     base  -> {base_extra[s]!r}")
print("  (💭 '12+57=' must NOT trigger: the guard requires a non-digit before a; the base model answers it alone)")

# False triggers: run plain text and check the circuit's logit change is exactly zero everywhere
with torch.no_grad():
    model(input_ids=tok(EVAL_TEXT + " " + " ".join(NEUTRAL), return_tensors="pt").input_ids.to(model.device), use_cache=False)
print(f"\nmax |Δz_circ| over every position of plain text = {graft.head.last_dz.abs().max().item()}   (📄 the paper reports 0.0)")

### 4.4 Natural-language retention, with and without norm compensation

📄 The paper reports next-token KL on held-out text: compensated Qwen 0.0008 vs. uncompensated 2.2725 (Tab. 5). Its circuits are large (Qwen K = 2,555 with d = 2,560, so γ ≈ 0.71). Our K = 65 is small, so we also run 🧪 **inert padding**: extra zero wires that bring the total width to 2d. That makes the norm-averaging effect as large as in the paper.

In [ ]:
def kl_now(): return kl(base_lp, logprobs(EVAL_TEXT))
rows = [("graft, compensated (γ)", "K + pad = %d" % K_WIRES, kl_now(),
         sum(generate(f"{a}+{b}=", 2 if a + b >= 10 else 1) == str(a + b) for a, b in PAIRS[::7]))]
graft.remove()
for comp, pad in [(False, 0), (True, cfg.hidden_size - K_WIRES), (False, cfg.hidden_size - K_WIRES)]:
    g2 = Graft(model, tok, compensate=comp, pad=pad); g2.calibrate(NEUTRAL); g2.build()
    acc = sum(generate(f"{a}+{b}=", 2 if a + b >= 10 else 1) == str(a + b) for a, b in PAIRS[::7])
    rows.append((f"graft, {'compensated (γ)' if comp else 'NO compensation'}", f"K + pad = {K_WIRES + pad}", kl_now(), acc))
    g2.remove()
print(f"{'variant':32s} {'width added':16s} {'KL(base||graft)':>16s}  arithmetic (15 pairs)")
for r in rows:
    print(f"{r[0]:32s} {r[1]:16s} {r[2]:16.5f}  {r[3]}/15")
print("\n💭 With fp32 on CPU, the compensated graft should be ~0 (rounding only). On GPU in bf16/4-bit it also includes "
      "the change from accumulating the residual stream in fp32.")
graft = Graft(model, tok); graft.calibrate(NEUTRAL); graft.build()   # leave the compensated graft installed

## 5. Limitations: how this toy differs from the paper

- 🧪 **Tiny program.** We compile 1-digit addition with 2 output tokens. 📄 The paper compiles 4–8 digit signed {+, −, ×}, Boolean (3-literal) and 5-bit Majority/Parity circuits, with Base-1000 canonical BPE emission (Granite) and an automated 5-stage compiler that reads the model config and tokenizer.
- 🧪 **Sidecar shifts instead of reprogrammed heads.** 📄 The paper routes tokens with Indefinite-RoPE self-sink attention (appended KV heads in Granite) and with memoryless GatedDeltaNet heads (A_log = 10, b_Δ = 10) and conv taps inside the real layers. We verify those primitives in isolation (2.6, 2.7) but implement the routing in the graft as an fp32 causal shift beside each layer. This assumes the shift is exact, which the paper proves for its constructions.
- 🧪 **Sidecar instead of a materialised d + K checkpoint.** Mathematically it is the block-diagonal extension. 📄 The paper ships both forms (dense patches and a factorized SparseSidecar, Tab. 3).
- 🧪 **KV cache off** during generation, because our shift reads earlier tokens' circuit wires.
- 🧪 **One-sided logit bumps.** 📄 The paper uses a differential (+254, −256) tied readout so tied embeddings cancel exactly.
- 💭 **Calibration margin and grid tails** (×/÷30, t ∈ [1e-3, 40]) are our choices; the paper gives explicit IEEE 754 tail bounds.
- **What the base model already knows.** Qwen3.5 adds single digits well on its own, so the accuracy here shows the graft is exact and harmless. It does not show a capability gain. The adder in 2.5 shows the mechanism scaling to 6 digits; 📄 the paper's 6-digit results on real models are in the appendix table.
- 📄 **Dual-use.** The paper notes that dormant, input-triggered weight-space payloads can be hidden in open-weight checkpoints and argues for structural checkpoint audits (Impact statement).

## 6. Exercises

1. **Grid spacing.** In 2.3, plot max |Q − 1| against Δ from 0.2 to 1.5. Where does it cross 1e-6? Why does a too-small window $[K_-, K_+]$ fail at extreme λ?
2. **Break the step.** In 2.4, lower `zmin` from 60 to 5, then 2. What happens to $\max|H|$ below the threshold? Compare with 📄 the condition z ≥ 2 and the bit-exact −0.0 condition z ≥ 89.
3. **Subtraction.** Extend the residual machine in 2.5 to compute A − B with borrows (step at −0.5).
4. **Kernel budget.** In 2.6, plot the best margin μ against the ℓ1 budget L. How many frequencies does the LP solution actually use?
5. **Two digits.** Extend the graft to `ab+cd=`: shifts up to δ = 5, a carry step, and a 3-token emission chain.
6. **Remove the guard.** Delete the `act` gate. Which non-arithmetic or multi-digit prompts now trigger?
7. **Compensation sweep.** In 4.4, sweep the padding from 0 to 4d with and without γ and plot the KL.

## 7. Appendix: numbers from the paper (📄 quoted from arXiv:2611.26450v1)

| claim | value | where |
|---|---|---|
| 6-digit arithmetic, N = 1,200, BF16, ALU-prefix accuracy | Granite 4.2 3B 99.17%; Qwen 3.5 4B 100.00% (100% with FP32 sidecar) | Abstract, Tab. 4 |
| Active non-zeros | Granite 1.07M–1.33M (≤ 0.036%); Qwen 2.25M–2.39M (≤ 0.057%) | Abstract |
| Mid-sentence resumption | 98%–100% | Abstract |
| LoRA baseline (12.8K samples) | 42.2% / 61.7%, 0% inline continuation | Abstract, Tab. 5 |
| Sidecar memory (A100) | 7.89 / 10.65 GiB; 4.60 / 8.14 GiB under INT8/INT4 + FP32 sidecar | Abstract, Tab. 3 |
| Long-context robustness | 14/14 pre-registered > 2K-token Tango splits (N = 4,392); 0/11,134 OOD triggers | Abstract |
| Boolean 3-literal circuits | 100% (N = 1,728) | Abstract |
| Mellin discretisation error | ε_disc(0.55) < 6.26 × 10⁻⁶; > 250× dynamic range | Thm 3.2, Abstract |
| Indefinite-RoPE kernel (θ = 10⁷, 64 freq.) | positive weights: S(δ) ≥ +0.61; signed: S(0) = +1, S(δ) ≤ −0.355 for 1 ≤ \|δ\| ≤ 1024 | Fig. 1b |
| Memoryless GDN | g_t = −e¹⁰ ln(1 + e¹⁰) < −2.20 × 10⁵ | Thm 3.3.2 |
| Norm compensation (Qwen) | NL KL 0.0008 vs 2.2725 uncompensated; ARC-C 92.7% vs 38.5% | Tab. 5 |
| Projection vs width extension | SVD projection 43×–3,800× higher KL (K = 1,140); uncompensated d + K 547× | Fig. 3a |
| Ablations | single-scale SwiGLU Arith 8.3% / 20.0%; stateful GDN (Qwen) 51.1% | Tab. 5 |